In [ ]:
#Fig 1

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as patches

# ==========================================
# setting
# ==========================================
plt.rcParams.update({
    "font.family": "serif",  
    "font.size": 14,
    "mathtext.fontset": "cm", 
})

fig = plt.figure(figsize=(14, 5.5))
ax = fig.add_subplot(111)
ax.set_axis_off()
ax.set_xlim(-1, 15)
ax.set_ylim(-3, 3)

N_nodes = 6

# ==========================================
# left: Dense Attention Graph 
# ==========================================
# random location
angles = np.linspace(0, 2*np.pi, N_nodes, endpoint=False)
left_cx, left_cy = 2.0, 0.0
radius = 1.8
x_left = left_cx + radius * np.cos(angles)
y_left = left_cy + radius * np.sin(angles)

# grey line
for i in range(N_nodes):
    for j in range(i+1, N_nodes):
        ax.plot([x_left[i], x_left[j]], [y_left[i], y_left[j]], color='gray', alpha=0.3, lw=1.5)

# grey node
ax.scatter(x_left, y_left, s=600, c='lightgray', edgecolor='dimgray', linewidth=2, zorder=10)

ax.text(left_cx, -2.5, "Deep Generative Model\n(Dense Attention Graph)", 
        ha='center', va='top', fontsize=16, fontweight='bold')

# ==========================================
# middle: The Bridge (Score = Force)
# ==========================================
arrow = patches.FancyArrowPatch((4.5, 0.0), (8.5, 0.0), 
                                arrowstyle='-|>,head_width=12,head_length=20', 
                                lw=5, color='black')
ax.add_patch(arrow)


ax.text(6.5, 0.8, "Score = Force", ha='center', va='bottom', fontsize=18, fontweight='bold', color='#d62728')
ax.text(6.5, 0.2, r"$\mathbf{s}_\theta(\mathbf{S}, t \to 0) \equiv -\beta \nabla H$", ha='center', va='bottom', fontsize=18)

# ==========================================
# right: Emergent Physical Hamiltonian
# ==========================================
x_right = np.linspace(10.0, 14.0, N_nodes)
y_right = np.zeros(N_nodes)

#  J1 
ax.plot(x_right, y_right, color='dimgray', lw=4, zorder=1)

# J2 
for i in range(N_nodes - 2):
    center_x = (x_right[i] + x_right[i+2]) / 2
    arc = patches.Arc((center_x, 0), width=(x_right[2]-x_right[0]), height=1.5, 
                      theta1=0, theta2=180, color='#2ca02c', ls='--', lw=2.5, zorder=2)
    ax.add_patch(arc)

# node A/B
colors_right =['#d62728', '#1f77b4', '#1f77b4', '#d62728', '#1f77b4', '#d62728']
labels_right =['A', 'B', 'B', 'A', 'B', 'A']
ax.scatter(x_right, y_right, s=600, c=colors_right, edgecolor='black', linewidth=2, zorder=5)

for i in range(N_nodes):
    ax.text(x_right[i], y_right[i]-0.4, labels_right[i], ha='center', va='top', fontsize=14, fontweight='bold')

# spin arrow
u =[0.5, -0.6, 0.2, -0.8, 0.7, -0.3]
v =[0.8, 0.8, -0.9, 0.6, 0.7, 0.9]
for i in range(N_nodes):
    norm = np.sqrt(u[i]**2 + v[i]**2)
    dx, dy = u[i]/norm * 0.7, v[i]/norm * 0.7
    ax.arrow(x_right[i], y_right[i], dx, dy, head_width=0.15, head_length=0.25, 
             fc='black', ec='black', lw=2.5, zorder=10)

ax.text(12.0, -2.5, "Underlying Physical Laws\n(Sparse Hamiltonian Basis)", 
        ha='center', va='top', fontsize=16, fontweight='bold')

plt.tight_layout()
plt.savefig("Fig1_Concept_Mapping.pdf", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
#Fig2

import torch
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import glob
from mini_af3_model import MiniAF3ScoreModel


def analyze_weight_decay(model_path="mini_af3_epoch_20.pt", test_dir="SO3_TestSet_2k_with_OracleForces"):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    model = MiniAF3ScoreModel(c_s=64, c_z=32, num_blocks=4).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    
    files = sorted(glob.glob(f"{test_dir}/chunk_*.npz"))
    

    max_r = 15
    decay_stats = {r:[] for r in range(1, max_r + 1)}
    
    with torch.no_grad():
        for f in files[:1]:  
            data = np.load(f, allow_pickle=True)
            seqs, spins_list = data['sequences'], data['spins']
            
            for seq_str, snapshots in zip(seqs, spins_list):
                N = len(seq_str)
                seq_tensor = torch.tensor([0 if c == 'A' else 1 for c in seq_str], dtype=torch.long).unsqueeze(0).to(device)
                mask = torch.ones((1, N), dtype=torch.bool).to(device)
                t_zero = torch.ones((1, 1)).to(device) * 1e-4 
                
                for snap in snapshots:
                    spin_tensor = torch.tensor(snap, dtype=torch.float32).unsqueeze(0).to(device)
                    

                    positions = torch.arange(N, device=device).unsqueeze(0).expand(1, N)
                    s = model.seq_emb(seq_tensor) + model.pos_emb(positions) + model.time_emb(t_zero).unsqueeze(1)
                    dist = torch.abs(positions.unsqueeze(2) - positions.unsqueeze(1)).clamp(max=99)
                    z = model.dist_emb(dist) + model.inner_prod_proj(torch.einsum('bid,bjd->bij', spin_tensor, spin_tensor).unsqueeze(-1))
                    
                    for block in model.blocks:
                        s, z = block(s, z, mask)
                        
                    W = model.weight_head(z).squeeze(-1)
                    W = (W + W.transpose(1, 2)) / 2.0
                    

                    diag_mask = 1.0 - torch.eye(N, device=device).unsqueeze(0)
                    W = W * diag_mask

                    
                    W_np = W.squeeze(0).cpu().numpy()
                    

                    for i in range(N):
                        for j in range(i+1, N):
                            r = abs(i - j)
                            if r <= max_r:
                                decay_stats[r].append(abs(W_np[i, j]))



    r_values = list(range(1, max_r + 1))
    mean_w = np.array([np.mean(decay_stats[r]) for r in r_values])
    perc_25 = np.array([np.percentile(decay_stats[r], 25) for r in r_values])
    perc_75 = np.array([np.percentile(decay_stats[r], 75) for r in r_values])
    

    lower_error = mean_w - perc_25
    upper_error = perc_75 - mean_w
    asymmetric_yerr = [lower_error, upper_error]
    

    plt.rcParams.update({
        "font.family": "serif",  
        "font.size": 14,
        "axes.labelsize": 16,
        "xtick.labelsize": 14,
        "ytick.labelsize": 14,
        "legend.fontsize": 12,
        "figure.figsize": (7, 5) 
    })
    

    fig, ax = plt.subplots()
    

    ax.errorbar(r_values, mean_w, yerr=asymmetric_yerr, fmt='-o', 
                color='#1f77b4', linewidth=2.5, markersize=8, 
                capsize=4, capthick=1.5, ecolor='#1f77b4', 
                label=r'Mean $|W_{ij}|$ (with IQR)')
    

    ax.axvline(2.5, color='#d62728', linestyle='--', linewidth=2, label='Bare Hamiltonian Cutoff ($r=2$)')
    

    ax.set_yscale('linear') 
    ax.set_xlabel("Sequence Distance $r = |i - j|$")
    ax.set_ylabel(r"Absolute Interaction Weight $|W_{ij}|$")
    ax.set_xticks(r_values[::2]) 
    ax.set_xlim(0.5, 15.5)
    

    ax.set_ylim(0.0, max(mean_w) * 1.15)
    ax.grid(True, linestyle=':', alpha=0.6)
    

    ax.legend(loc='upper right', framealpha=0.95)
    

    axins = ax.inset_axes([0.45, 0.15, 0.50, 0.48]) 
    
    axins.errorbar(r_values, mean_w, yerr=asymmetric_yerr, fmt='-o', 
                   color='#1f77b4', linewidth=1.5, markersize=4, 
                   capsize=3, capthick=1.0, ecolor='#1f77b4')
    axins.axvline(2.5, color='#d62728', linestyle='--', linewidth=1.5)
    

    axins.set_yscale('log')
    axins.set_xlim(0.5, 15.5)
    axins.set_xticks([1, 5, 10, 15])
    axins.tick_params(axis='both', which='major', labelsize=10)
    axins.set_title("Log Scale", fontsize=11, pad=4)
    axins.grid(True, linestyle=':', alpha=0.4)
    

    plt.tight_layout()
    plt.savefig("Fig2_Weight_Decay.pdf", dpi=300, bbox_inches='tight')
    plt.show()

if __name__ == '__main__':
    analyze_weight_decay()

In [ ]:
#Fig S1
import numpy as np
import matplotlib.pyplot as plt
import glob
from tqdm import tqdm

print("🔍 scanning all data...")
train_files = sorted(glob.glob("SO3_TrainSet_40k/chunk_*.npz"))
test_files = sorted(glob.glob("SO3_TestSet_2k_with_OracleForces/chunk_*.npz"))

if not train_files or not test_files:
    print("❌ cannot find data, check path!")
else:

    lengths = []
    dot_products = {'AA': [], 'BB':[], 'AB': []}
    force_magnitudes = []
    spins_list_global =[] 
    
    print(f"📥 loading ({len(train_files)} Train Chunks)...")
    for file in tqdm(train_files, desc="Train"):
        data = np.load(file, allow_pickle=True)
        seqs, spins_list = data['sequences'], data['spins']
        
        for seq, spins in zip(seqs, spins_list):
            lengths.append(len(seq))
            dot_vals = np.sum(spins[:, :-1, :] * spins[:, 1:, :], axis=-1)
            for i in range(len(seq) - 1):
                pair = "".join(sorted([seq[i], seq[i+1]]))
                dot_products[pair].extend(dot_vals[:, i])
                
    print(f"📥 loading ({len(test_files)} Test Chunks)...")
    for file in tqdm(test_files, desc="Test"):
        data = np.load(file, allow_pickle=True)
        forces_list, spins_list = data['tangent_forces'], data['spins']
        
 
        spins_list_global.extend(spins_list)
        
        for forces in forces_list:
            mags = np.linalg.norm(forces, axis=-1)
            force_magnitudes.extend(mags.flatten())
            
    print("✅ Data perfectly loaded into memory!")



In [ ]:
#Fig S1
print("🎨 painting...")

from matplotlib.gridspec import GridSpec


plt.rcParams.update({
    "font.family": "serif",  
    "font.size": 13,
    "axes.labelsize": 14,
    "axes.titlesize": 15,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
    "legend.fontsize": 11,
})

fig = plt.figure(figsize=(12, 9))

gs = GridSpec(2, 2, figure=fig, wspace=0.25, hspace=0.35)

# [Subplot 1] Sequence Lengths
ax1 = fig.add_subplot(gs[0, 0])
ax1.hist(lengths, bins=np.arange(19.5, 51.5, 1), color='#4c72b0', edgecolor='black', linewidth=0.8, alpha=0.85)
ax1.set_title(r"$\mathbf{(a)}$ Sequence Length Distribution", loc='left')
ax1.set_xlabel("Chain Length $N$")
ax1.set_ylabel("Frequency")
ax1.grid(axis='y', linestyle=':', alpha=0.6)

#[Subplot 2] Local Frustration
ax2 = fig.add_subplot(gs[0, 1])
colors = {'AA': '#d62728', 'BB': '#1f77b4', 'AB': '#2ca02c'}
labels = {'AA': 'A-A (Rigid)', 'BB': 'B-B (Flexible)', 'AB': 'A-B (Moderate)'}
for pair, dots in dot_products.items():
    ax2.hist(dots, bins=80, range=(-0.7, 1.0), alpha=0.6, 
             histtype='stepfilled', edgecolor='black', linewidth=0.5,
             label=f"Bond: {labels[pair]}", color=colors[pair], density=True)
ax2.set_title(r"$\mathbf{(b)}$ Local Frustration by Bond Type", loc='left')
ax2.set_xlabel(r"Nearest-Neighbor Dot Product ($\mathbf{S}_i \cdot \mathbf{S}_{i+1}$)")
ax2.set_ylabel("Probability Density")
ax2.legend(loc='upper left', framealpha=0.9)
ax2.axvline(1.0, color='black', linestyle='--', alpha=0.4, linewidth=1.5)
ax2.grid(True, linestyle=':', alpha=0.6)

# [Subplot 3] Theoretical force distribution 
ax3 = fig.add_subplot(gs[1, 0])
ax3.hist(force_magnitudes, bins=80, range=(0, np.percentile(force_magnitudes, 99.9)), 
         color='#9467bd', edgecolor='black', linewidth=0.5, alpha=0.85)
ax3.set_title(r"$\mathbf{(c)}$ Tangent Force Magnitudes", loc='left')
ax3.set_xlabel(r"Force Magnitude $|\mathbf{F}_{\text{tan}}|$")
ax3.set_ylabel("Frequency")
ax3.grid(axis='y', linestyle=':', alpha=0.6)

# [Subplot 4] Emergent 3D Conformations 

ax4_base = fig.add_subplot(gs[1, 1])
ax4_base.set_axis_off()
ax4_base.set_title(r"$\mathbf{(d)}$ Emergent 3D Conformations", loc='left', pad=10)


gs_inner = gs[1, 1].subgridspec(2, 2, wspace=0.0, hspace=0.0)

num_chains = 4
sample_indices = np.linspace(0, len(spins_list_global) - 1, num_chains).astype(int)
chain_colors =['#ff7f0e', '#2ca02c', '#d62728', '#9467bd']


for i, (idx, color) in enumerate(zip(sample_indices, chain_colors)):

    row = i // 2
    col = i % 2
    ax_mini = fig.add_subplot(gs_inner[row, col], projection='3d')
    
    spins_3d = spins_list_global[idx][0] 
    coords = np.vstack([[0, 0, 0], np.cumsum(spins_3d, axis=0)])
    

    center_of_mass = np.mean(coords, axis=0)
    coords = coords - center_of_mass
    
    ax_mini.plot(coords[:,0], coords[:,1], coords[:,2], color=color, 
                 linewidth=2.0, marker='o', markersize=4.0, 
                 markeredgecolor='black', markeredgewidth=0.3, alpha=0.9)
                 

    mid_x = (coords[:,0].max() + coords[:,0].min()) * 0.5
    mid_y = (coords[:,1].max() + coords[:,1].min()) * 0.5
    mid_z = (coords[:,2].max() + coords[:,2].min()) * 0.5
    
    max_range = np.array([coords[:,0].max() - coords[:,0].min(), 
                          coords[:,1].max() - coords[:,1].min(), 
                          coords[:,2].max() - coords[:,2].min()]).max() / 2.0
    
    margin = max_range * 1.05
    ax_mini.set_xlim(mid_x - margin, mid_x + margin)
    ax_mini.set_ylim(mid_y - margin, mid_y + margin)
    ax_mini.set_zlim(mid_z - margin, mid_z + margin)
    

    ax_mini.view_init(elev=20, azim=45 + i*25) 
    

    try:
        ax_mini.set_box_aspect(None, zoom=1.8)
    except:
        ax_mini.dist = 7
        
    ax_mini.set_axis_off()

plt.tight_layout()
plt.savefig("FigS1_Dataset_Overview.pdf", dpi=300, bbox_inches='tight')
plt.show()
print("✅ finished!")